# 03 FSD, Robotaxi and Optimus Options

FSD software can sit inside segment cash flow. Robotaxi and Optimus stay
as probability-weighted option values until filings or operating data
support base-case revenue recognition.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/tesla_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/tesla_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
options = pd.read_csv(DATA_DIR / "option_scenarios.csv")
fsd = (
    assumptions[assumptions["segment"].eq("FSD")]
    .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
    .reindex(["bear", "base", "bull"])
)
fsd

In [ ]:
fsd_case = fsd.assign(
    ebitda_usd_bn=lambda df: df["2030 revenue"] * df["EBITDA margin"],
    ev_revenue_multiple=[5.0, 10.0, 15.0],
    ev_ebitda_multiple=[18.0, 26.0, 35.0],
)
fsd_case["selected_value_usd_bn"] = (
    fsd_case["2030 revenue"] * fsd_case["ev_revenue_multiple"]
    + fsd_case["ebitda_usd_bn"] * fsd_case["ev_ebitda_multiple"]
) / 2
fsd_case

In [ ]:
option_values = {}
for option_name, group in options.groupby("option"):
    scenarios = [
        OptionScenario(row.case, row.probability, row.value_usd_bn)
        for row in group.itertuples(index=False)
    ]
    option_values[option_name] = probability_weighted_option_value(scenarios)
option_values

In [ ]:
fsd_attach_grid = build_sensitivity_grid(
    row_values=[0.10, 0.20, 0.30, 0.40, 0.50],
    column_values=[50, 75, 100, 125, 150],
    formula=lambda attach_rate, monthly_fee: (
        30_000_000 * attach_rate * monthly_fee * 12 / 1_000_000_000
    ),
    row_name="FSD attach rate",
    column_name="Monthly fee ($)",
)
fsd_attach_grid

In [ ]:
robotaxi_grid = build_sensitivity_grid(
    row_values=[0.10, 0.25, 0.40, 0.55],
    column_values=[250_000, 500_000, 1_000_000, 2_000_000],
    formula=lambda success_probability, active_units: (
        success_probability * active_units * 60_000 / 1_000_000_000
    ),
    row_name="Robotaxi success probability",
    column_name="Active robotaxi units",
)
robotaxi_grid